## 1. Import Requirements

# Master Shear-Wave Splitting Workflow for Axial Seamount

This notebook provides a complete, clean workflow from raw earthquake catalog and waveform data to shear-wave splitting analysis results. The workflow follows proper sequencing and includes all necessary quality control measures. 

Instead of using catalog from Wilcock and Zhang or ML DD, we use the nlloc file for all stations from Christian's results.

## Workflow Overview

1. **Data Loading & Initial Setup** - Load earthquake catalog and station metadata
2. **Extended Time Window Creation** - Create proper time windows for waveform retrieval
3. **Waveform Data Retrieval** - Download seismic data with extended windows
4. **Quality Control Filters** - P-wave rectilinearity, SNR, and incidence angle filtering
5. **Geometric Calculations** - Back-azimuth and distance calculations
6. **Shear-Wave Splitting Analysis** - Dynamic parameter estimation and SWSPy analysis
7. **Results Processing & Visualization** - Compile and visualize splitting parameters

## Key Improvements
- Extended catalog creation moved to proper early position
- Updated P-wave polarization analysis for true incidence angles
- Integrated SNR calculations with proper S-wave timing
- Clean separation of quality control steps

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Load 2018 earthquake test catalog - ML DD
#catalog = pd.read_csv('2018_eq_catalog.csv')

# Load Baillard nonlinloc catalog
catalog = pd.read_csv('AXIAL.PHASE.FINAL_3D_V2.csv')

#Load station information from Christian's data
stations_file = '../data/stations_axial.llz'
#Read llz file - reads like a text file with space delimiter
stations_df = pd.read_csv(stations_file, delim_whitespace=True, header=None, names=['Longitude (°W)', 'Latitude (°N)', 'Elevation (m)', 'Station ID'])
# Convert elevation column to m from km
stations_df['Elevation (m)'] = stations_df['Elevation (m)']*1000

print(f"Stations in catalog: {catalog['station'].value_counts()}")

In [ ]:


# Look at AXEC1 station
axec1_catalog = catalog[catalog['station'] == 'AXEC1'].copy()

# Reset index to ensure clean indexing
#axas1_catalog = axas1_catalog.reset_index(drop=True)
axec1_catalog = axec1_catalog.reset_index(drop=True)


axec1_catalog['datetime'] = pd.to_datetime(axec1_catalog['datetime'])
axec1_catalog = axec1_catalog[(axec1_catalog['datetime'] >= '2015-04-14') & (axec1_catalog['datetime'] < '2015-06-01')].copy()

# Select first 100 events for testing
#test_catalog_100 = axas2_catalog.head(100).copy()


#print(f"Total AXAS2 events in catalog: {len(axas2_catalog)}")
print(f"Total AXEC1 events in catalog: {len(axec1_catalog)}")

#print(f"Test catalog created with first {len(test_catalog_100)} AXAS2 events")
print(f"\nDate range of test catalog:")
print(f"Start: {axec1_catalog['datetime'].min()}")
print(f"End: {axec1_catalog['datetime'].max()}")

display(axec1_catalog)

In [ ]:
# Downsample catalog by a factor of 4 - take every fourth event
#test_catalog = axas2_catalog.iloc[::4].copy()

test_catalog = axec1_catalog.copy()

print(f"\nDate range of test catalog:")
print(f"Start: {test_catalog['datetime'].min()}")
print(f"End: {test_catalog['datetime'].max()}")

In [ ]:
# First, reformat the datetime strings to add 'T' separator
test_catalog['p_time'] = test_catalog['p_time'].str.replace(' ', 'T', regex=False)
test_catalog['s_time'] = test_catalog['s_time'].str.replace(' ', 'T', regex=False)
test_catalog['datetime'] = test_catalog['datetime'].astype(str).str.replace(' ', 'T', regex=False)

# Now convert to pandas Timestamp with UTC timezone
test_catalog['p_time'] = pd.to_datetime(test_catalog['p_time'], utc=True, format='ISO8601')
test_catalog['s_time'] = pd.to_datetime(test_catalog['s_time'], utc=True, format='ISO8601')
test_catalog['datetime'] = pd.to_datetime(test_catalog['datetime'], utc=True, format='ISO8601')

# Convert to UTCDateTime
test_catalog['p_time'] = test_catalog['p_time'].apply(lambda x: UTCDateTime(x))
test_catalog['s_time'] = test_catalog['s_time'].apply(lambda x: UTCDateTime(x))
test_catalog['datetime'] = test_catalog['datetime'].apply(lambda x: UTCDateTime(x))

print("Successfully converted to UTCDateTime")
print(f"Sample p_time: {test_catalog['p_time'].iloc[0]}")

In [ ]:
catalog = test_catalog.copy()

## 3. Extended Time Window Creation

This step creates extended time windows for waveform retrieval. This is critical for proper analysis and must happen early in the workflow, before any quality control that depends on waveform data.

In [ ]:
# Create extended time windows for proper waveform analysis
print("Creating extended time windows for waveform retrieval...")

# Apply extended windowing
extended_catalog = create_extended_catalog(catalog, pre_p_time=1.0, post_s_time=2.0)

print(f"Extended catalog created with {len(extended_catalog)} events")
print(f"Time windows: {extended_catalog['total_duration'].iloc[0]} seconds total")
print(f"Pre-event: {extended_catalog['pre_p_sec'].iloc[0]}s, Post-event: {extended_catalog['post_s_sec'].iloc[0]}s")
# Display sample of extended timing
print("\nSample timing windows:")
sample_cols = ['id', 'datetime', 'starttime', 'endtime', 'total_duration']
display(extended_catalog[sample_cols])

In [ ]:
# Remove leading 'OO' from station names
extended_catalog['station'] = extended_catalog['station'].str.replace('OO', '', regex=False)

In [ ]:
display(extended_catalog)

## 4. Waveform Data Retrieval

This section retrieves seismic waveform data using the extended time windows. We'll load the existing trace data and organize it for processing.

In [ ]:
test_catalog = extended_catalog

In [ ]:
# Replace catalog id with index
test_catalog['id'] = test_catalog.index

test_catalog['mag'] = 0.0

In [ ]:
def get_station_traces_batch(df, filename, starttime, endtime, station_id, batch_size=250):
    """
    Fast bulk retrieval of waveform data with intelligent batched fallback.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with event information
    filename : str
        Output filename (without extension)
    starttime : str
        Column name for start time
    endtime : str
        Column name for end time
    station_id : str
        Column name for station ID
    
    Returns:
    --------
    obspy.Stream : All retrieved traces
    
    Performance:
    - Bulk success: ~10-15 seconds for 250 events
    - Batched fallback: ~30-60 seconds (250 events per batch)
    - Individual fallback: Only for failed batches
    """
    from obspy.clients.fdsn import Client
    from obspy.core.utcdatetime import UTCDateTime
    from obspy import Stream
    import time
    
    client = Client("IRIS")
    all_traces = Stream()

    # BATCHED- 250 events per batch
    batch_size = batch_size
    total_batches = (len(df) + batch_size - 1) // batch_size
    
    for batch_idx in range(total_batches):
        batch_start_idx = batch_idx * batch_size
        batch_end_idx = min(batch_start_idx + batch_size, len(df))
        batch_df = df.iloc[batch_start_idx:batch_end_idx]
        
        print(f"\n{'─'*60}")
        print(f"BATCH {batch_idx + 1}/{total_batches}")
        print(f"Events {batch_start_idx + 1} to {batch_end_idx} ({len(batch_df)} events)")
        print(f"{'─'*60}")
        
        # Build bulk request for this batch
        batch_bulk_list = []
        for _, row in batch_df.iterrows():
            t_start = UTCDateTime(row[str(starttime)]) - 0.5
            t_final = UTCDateTime(row[str(endtime)]) + 0.5
            current_station = row[str(station_id)]
            
            if current_station == 'AXEC1':
                batch_bulk_list.append(('OO', 'AXEC1', '', 'EHE', t_start, t_final))
                batch_bulk_list.append(('OO', 'AXEC1', '', 'EHN', t_start, t_final))
                batch_bulk_list.append(('OO', 'AXEC1', '', 'EHZ', t_start, t_final))
        
        # Try batch bulk request
        batch_start_time = time.time()
        batch_stream = client.get_waveforms_bulk(batch_bulk_list)
        batch_elapsed = time.time() - batch_start_time
        
        all_traces += batch_stream
        
        print(f"✓ Batch {batch_idx + 1} SUCCESS: {len(batch_stream)} traces in {batch_elapsed:.1f}s")
        print(f"  Expected: {len(batch_bulk_list)}, Retrieved: {len(batch_stream)}")
        
        if len(batch_stream) < len(batch_bulk_list):
            missing = len(batch_bulk_list) - len(batch_stream)
            print(f"  ⚠ Warning: {missing} traces missing from this batch")
            

            
            print(f"\n{'='*60}")
            print(f"BATCHED FALLBACK COMPLETE")
            print(f"{'='*60}")
            print(f"Total traces retrieved: {len(all_traces)}")
    
    # Save results
    if all_traces:
        print(f"\nSaving {len(all_traces)} traces to {filename}.mseed...")
        all_traces.write(str(filename) + ".mseed", format="MSEED")
        print(f"✓ File saved successfully")
        
        # Summary statistics
        print(f"\n{'='*60}")
        print(f"RETRIEVAL SUMMARY")
        print(f"{'='*60}")
        print(f"Total events processed: {len(df)}")
        print(f"Total traces retrieved: {len(all_traces)}")
        print(f"Expected traces (max): {len(df) * 3}")
        print(f"Success rate: {len(all_traces)/(len(df)*3)*100:.1f}%")
        
    else:
        print(f"\n⚠ WARNING: No traces retrieved!")
    
    return all_traces

In [ ]:
# Retrieve waveforms for all events in the test catalog using get_all_traces function
print("Retrieving waveforms for all events in the test catalog...")
waveforms = get_station_traces_batch(test_catalog, 'axial_nonlinloc_apr_14_jun_01_axec1', 'starttime', 'endtime', 'station', batch_size=100)

In [ ]:
# Load waveforms from mseed file with obspy
waveforms_file = 'axial_nonlinloc_apr_14_jun_01_axec1.mseed'
waveforms = obspy.read(waveforms_file)

In [ ]:
# Associate waveforms with events in the catalog
print("Organizing waveforms by events...")
waveform_dict = organize_stream_by_events(waveforms, test_catalog)

In [ ]:
# Organize waveforms by event ID
print("Organizing waveforms by event ID...")
organized_waveforms = organize_waveform_data(waveform_dict, test_catalog)

In [ ]:
# Format s_arrival_time and p_arrival_time as difference between arrival times and origin time
print("Formatting s_arrival_time and p_arrival_time as differences from origin time...")
for eid in organized_waveforms.keys():
    organized_waveforms[eid]['s_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 's_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))
    organized_waveforms[eid]['p_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'p_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))

In [ ]:
# For all traces in organized_waveforms, taper and filter in-place
print("Tapering and filtering all traces in organized_waveforms...")
events_to_remove = []
try:
    for eid in organized_waveforms.keys():
        for tr in organized_waveforms[eid]['traces']:
            tr.detrend("linear") # to avoid weird start and end amplitudes
            tr.taper(max_percentage=0.05, type='hann')
            tr.filter('bandpass', freqmin=5.0, freqmax=40.0)
except Exception as e:
    print(f"Error during waveform processing: {e}")
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
    print(f"Events with issues: {events_to_remove}")

print("Waveform retrieval and organization complete.")

In [ ]:
# Find streams that are NoneType and remove from organized_waveforms
print("Checking for NoneType streams in organized_waveforms...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
if events_to_remove:
    print(f"Removing {len(events_to_remove)} events with NoneType streams: {events_to_remove}")
    for eid in events_to_remove:
        del organized_waveforms[eid]
else:
    print("No NoneType streams found in organized_waveforms.")

In [ ]:
# Remove duplicate traces from organized_waveforms
print("Checking for and removing duplicate traces in organized_waveforms...")

for eid in organized_waveforms.keys():
    # Get the stream for this event
    st = organized_waveforms[eid]['traces']
    
    # Check if there are duplicates
    try:
        if type(st) == type(None):
            print(f"Event {eid}: No traces found (NoneType)")
            continue

        else:
            if len(st) > 3:
                print(f"Event {eid}: Found {len(st)} traces (expected 3)")
                
                # Create a new stream with unique traces based on channel code
                unique_traces = {}
                for tr in st:
                    channel = tr.stats.channel
                    # Keep the first occurrence of each channel
                    if channel not in unique_traces:
                        unique_traces[channel] = tr
                
                # Replace the stream with deduplicated traces
                organized_waveforms[eid]['traces'] = obspy.Stream(traces=list(unique_traces.values()))
                print(f"  Reduced to {len(organized_waveforms[eid]['traces'])} unique traces")
    except Exception as e:
        print(f"Error processing event {eid}: {e}")
        organized_waveforms[eid]['traces'] = st[:3]  # Fallback to first 3 traces if error occurs

# Verify the results
print("\nVerification of trace counts after deduplication:")
trace_counts = {}
for eid in organized_waveforms.keys():
    count = len(organized_waveforms[eid]['traces'])
    trace_counts[count] = trace_counts.get(count, 0) + 1

print(f"Events with 3 traces: {trace_counts.get(3, 0)}")
if any(k != 3 for k in trace_counts.keys()):
    print("Events with unexpected trace counts:")
    for count, num_events in trace_counts.items():
        if count != 3:
            print(f"  {num_events} events with {count} traces")
else:
    print("All events have exactly 3 traces (E, N, Z)")

In [ ]:
# Remove events that do not have exactly 3 traces
print("\nRemoving events that do not have exactly 3 traces...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if len(organized_waveforms[eid]['traces']) != 3:
        events_to_remove.append(eid)

    # also remove events with any trace that has zero length (indicating a retrieval issue) or empty traces
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

for eid in events_to_remove:
    del organized_waveforms[eid]

## 5. Quality Control Pipeline

This section implements comprehensive quality control measures including P-wave rectilinearity analysis, signal-to-noise ratio calculations, and incidence angle filtering.

In [ ]:
# Define quality control thresholds
QC_THRESHOLDS = {
    'min_snr': 2.0,           # Minimum S-wave signal-to-noise ratio
    'min_rectilinearity': 0.7, # Minimum P-wave rectilinearity
    'max_incidence': 30.0,     # Maximum incidence angle (degrees)
}

print("Quality control functions loaded successfully")
print(f"QC Thresholds: {QC_THRESHOLDS}")

In [ ]:
# Check that all traces for same event have same length, and remove events that do not meet this criterion
print("Checking that all traces for the same event have the same length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    trace_lengths = [tr.stats.npts for tr in organized_waveforms[eid]['traces']]
    if len(set(trace_lengths)) != 1:
        print(f"Event {eid} has traces of different lengths: {trace_lengths}, marking for removal")
        events_to_remove.append(eid)

In [ ]:
# Check if any traces are length zero, and if so mark those events for removal
print("Checking for traces with zero length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

In [ ]:
if events_to_remove:
    print(f"Removing {len(events_to_remove)} events that do not have traces of the same length or have zero-length traces: {events_to_remove}")
    for eid in events_to_remove:
        del organized_waveforms[eid]
else:
    print("All events have traces of the same length and no zero-length traces found.")

In [ ]:
# Calculate quality control metrics for organized waveforms
print("Calculating quality control metrics for organized waveforms...")

# 1. Calculate S-wave SNR
organized_waveforms = calculate_snr_for_organized_waveforms(organized_waveforms)

# 2. Calculate geographic back-azimuth, for coordinate rotation later
organized_waveforms = calculate_back_azimuth_for_organized_waveforms(organized_waveforms, stations_df)

# 3. Calculate incidence angle
organized_waveforms = calculate_incidence_angle_eigenvalue_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

#4. Calculate P-wave rectilinearity
organized_waveforms = calculate_rectilinearity_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

In [ ]:
# Define passing_waveforms as those that meet all QC thresholds
passing_waveforms = apply_quality_control(organized_waveforms, QC_THRESHOLDS)

In [ ]:
# Save the passing_waveforms
metadata_df = save_passing_waveforms(passing_waveforms, output_dir='passing_waveforms_data_nonlinloc_axec1')
display(metadata_df.head())

In [ ]:
# Reload the data
passing_waveforms_reloaded = load_passing_waveforms(output_dir='passing_waveforms_data_nonlinloc_axec1')

# Verify the reload worked correctly
print(f"Reloaded events: {len(passing_waveforms_reloaded)}")
print(f"\nSample reloaded event (ID: {list(passing_waveforms_reloaded.keys())[0]}):")
sample_event = passing_waveforms_reloaded[list(passing_waveforms_reloaded.keys())[0]]
print(f"  Station: {sample_event['station']}")
print(f"  Origin time: {sample_event['origin_time']}")
print(f"  Number of traces: {len(sample_event['traces'])}")
print(f"  Back azimuth: {sample_event['back_azimuth']:.2f}°")

## 6. Shear-Wave Splitting Analysis

This section implements the core shear-wave splitting analysis using SWSPy with dynamic parameter estimation and comprehensive quality assessment.

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start= 2, last_window_start=1, 
                                                         first_window_end=1.8, last_window_end=2.2, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

save_results_csv(results_swspy, file_name='splitting_results_axec1_apr_14_jun_01_nonlinloc')